# Traitement automatisé — toutes les visites (V0, V1, V3, V5, Vc)

Exécutez ce notebook de haut en bas.  
La fonction `traiter_visite(v)` applique exactement les mêmes traitements que le notebook V0 original,  
en paramétrant le numéro de visite.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce

---
## Utilitaire : nettoyage des codes manquants (".D", ".A", ".K", ".F", "D...")

In [ ]:
def nettoyer_codes_manquants(df, prefixes=("." ,)):
    """
    Repère toutes les valeurs texte commençant par l'un des préfixes
    dans les colonnes object du DataFrame, puis les remplace par NaN.
    Retourne le DataFrame nettoyé (inplace).
    """
    text_cols = df.select_dtypes(include="object").columns
    all_unique = set()
    for col in text_cols:
        all_unique.update(df[col].dropna().unique())

    codes = [v for v in all_unique
             if any(str(v).startswith(p) for p in prefixes)]
    if codes:
        print(f"  → codes remplacés par NaN : {codes}")
    return df.replace(codes, np.nan)

---
## Fonction principale : `traiter_visite(v, vc=False)`

In [ ]:
def traiter_visite(v, vc=False):
    """
    Lit Output/version_2/V{v}.xlsx et applique le même traitement
    que le notebook manuel V.

    Paramètres
    ----------
    v   : int ou str  — numéro de visite : 0, 1, 3, 5 ou "c" (pour Vc)
    vc  : bool        — si True, traite uniquement les feuilles communes Vc

    Retourne
    --------
    dict {str: pd.DataFrame}
    """
    chemin = f"Output/version_2/V{v}.xlsx"
    print(f"\n{'='*60}")
    print(f"  TRAITEMENT  V{v}  —  {chemin}")
    print(f"{'='*60}")

    # ==================================================================
    # FEUILLES COMMUNES  (présentes dans toutes les visites, y compris Vc)
    # ==================================================================
     
    # ──  ──────────────────────────────────────────────────
    df_V = pd.read_excel(chemin, sheet_name=f"V{v}")
    # print(f"\nFeuille     : {V{v}.shape}")
    df_V = nettoyer_codes_manquants(df_V)


    # df_base = pd.read_excel(chemin, sheet_name=f"df_v{v}")
    # df_base.drop(columns=["POIDS_NR", "TAILLE_NR", "TITRE"], inplace=True)
    # df_V = pd.merge(df_dates, df_base, on="SUBJID", how="left")
    # df_V.insert(0, "VISITE", v)
    # df_V.drop("INIT_PAT", axis=1, inplace=True)
    # print(f"\ndf_V (après fusion)     : {df_V.shape}")


    # ── LEDD (Feuil1) ─────────────────────────────────────────────────
    df_LEDD = pd.read_excel(chemin, sheet_name="LEDD")
    # print(f"Feuille Feuil1 (LEDD)   : {df_feuil1.shape}")
    # df_feuil1.rename(columns={"Subject Identifier for the Study": "SUBJID"}, inplace=True)
    # df_LEDD = df_feuil1.copy()
    # df_LEDD.drop("Num", axis=1, inplace=True)
    # df_LEDD.insert(0, "VISITE", v)
    # Calcul ledd_tot à partir de la somme (colonnes 2 → avant-dernière)
    df_LEDD["somme_calc"] = df_LEDD.iloc[:, 2:-1].sum(axis=1, skipna=True)
    df_LEDD["statut"] = np.where(
        np.isclose(df_LEDD["somme_calc"], df_LEDD["ledd_tot"], atol=0.01),
        "ok", "différent"
    )
    df_LEDD["ledd_tot"] = df_LEDD["somme_calc"]
    df_LEDD.drop(columns=["somme_calc", "statut"], inplace=True)
    df_Total_LEDD = df_LEDD[["SUBJID", "ledd_tot"]]
    print(f"df_LEDD                 : {df_LEDD.shape}")

    # ── LEDD_info (feuille LEDD complète) ─────────────────────────────
    # df_ledd_raw = pd.read_excel(chemin, sheet_name="LEDD")
    # print(f"Feuille LEDD            : {df_ledd_raw.shape}")
    # df_LEDD_info = df_ledd_raw.iloc[:, :-7].copy()
    # df_LEDD_info.drop("v", axis=1, inplace=True)
    # df_LEDD_info.rename(columns={"Subject Identifier for the Study": "SUBJID"}, inplace=True)
    # df_LEDD_info.insert(0, "VISITE", v)
    # df_LEDD_info.drop("Num",   axis=1, inplace=True)
    # df_LEDD_info.drop("Visit", axis=1, inplace=True)
    # print(f"df_LEDD_info            : {df_LEDD_info.shape}")

    # ── PSYCHOTROPES ──────────────────────────────────────────────────
    df_PSYCHOTROPES = pd.read_excel(chemin, sheet_name="PSYCHOTROPES")
    print(f"\nFeuille PSYCHOTROPES    : {df_PSYCHOTROPES.shape}")
    # df_PSYCHOTROPES.insert(0, "VISITE", v)
    # df_PSYCHOTROPES.drop(columns=["NUM", "Unnamed: 3"], inplace=True)
    # df_PSYCHOTROPES.drop(columns=["VISIT", "INIT_PAT"], inplace=True)
    df_PSYCHOTROPES = nettoyer_codes_manquants(df_PSYCHOTROPES)
    print(f"df_PSYCHOTROPES         : {df_PSYCHOTROPES.shape}")

    # ── AUTRE_PARKINSON ───────────────────────────────────────────────
    df_AUTRE_PARKINSON = pd.read_excel(chemin, sheet_name="AUTRE_PARKINSON")
    # print(f"\nFeuille AUTRE_PARKINSON : {df_AUTRE_PARKINSON.shape}")
    # df_AUTRE_PARKINSON.insert(0, "VISITE", v)
    # df_AUTRE_PARKINSON.drop(columns=["NUM", "Unnamed: 3"], inplace=True)
    # df_AUTRE_PARKINSON.drop(columns=["VISIT", "INIT_PAT"], inplace=True)
    df_AUTRE_PARKINSON = nettoyer_codes_manquants(df_AUTRE_PARKINSON)
    print(f"df_AUTRE_PARKINSON      : {df_AUTRE_PARKINSON.shape}")

    # ── CONSO_SPECIFIQUE ──────────────────────────────────────────────
    df_CONSO_SPECIFIQUE = pd.read_excel(chemin, sheet_name="CONSO_SPECIFIQUE")
    # print(f"\nFeuille CONSO_SPECIFIQUE: {df_CONSO_SPECIFIQUE.shape}")
    # df_CONSO_SPECIFIQUE.insert(0, "VISITE", v)
    # df_CONSO_SPECIFIQUE.drop(
    #     columns=["NUM", "VISIT_NOM", "NUM_CENTRE", "NUM_PAT", "INIT_PAT"], inplace=True
    # )
    # df_CONSO_SPECIFIQUE.drop("VISIT", axis=1, inplace=True)
    df_CONSO_SPECIFIQUE = nettoyer_codes_manquants(df_CONSO_SPECIFIQUE)
    print(f"df_CONSO_SPECIFIQUE     : {df_CONSO_SPECIFIQUE.shape}")

    # ==================================================================
    # CAS SPÉCIAL  →  Vc  (uniquement les feuilles communes)
    # ==================================================================
    if vc:
        return {
            f"V{v}"             : df_V,
            "LEDD"             : df_LEDD,
            "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
            "PSYCHOTROPES"     : df_PSYCHOTROPES,
            "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
        }

    # ==================================================================
    # FEUILLES SPÉCIFIQUES  V → V5
    # ==================================================================

    # # ── V (données générales) ─────────────────────────────────────────
    # df_base = pd.read_excel(chemin, sheet_name=f"df_v{v}")
    # df_base.drop(columns=["POIDS_NR", "TAILLE_NR", "TITRE"], inplace=True)
    # df_V = pd.merge(df_dates, df_base, on="SUBJID", how="left")
    # df_V.insert(0, "VISITE", v)
    # df_V.drop("INIT_PAT", axis=1, inplace=True)
    # print(f"\ndf_V (après fusion)     : {df_V.shape}")

    # ── UPDRSIII ──────────────────────────────────────────────────────
    # V et V1 : structure identique (provient du même fichier source)
    # V3 et V5 : structure différente (fichier UPDRSIII_COMPLET_V3_V5)
    # if v in (0, 1):
    #     df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
    #     print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")
    #     df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=(".",))
    #     df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=("D",))
    #     df_Total_UPDRSIII = df_UPDRSIII.iloc[:, [1] + list(range(-7, 0))]
    #     print(f"df_UPDRSIII             : {df_UPDRSIII.shape}")
    
    if v in (0, 1):
        df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
        print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")
    
        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=(".",))
        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=("D",))

        # nombre de colonnes à prendre à la fin
        n_last = 7 if v == 0 else 4

        df_Total_UPDRSIII = df_UPDRSIII.iloc[:, [1] + list(range(-n_last, 0))]

        print(f"df_UPDRSIII             : {df_UPDRSIII.shape}")

    elif v in (3, 5):
        # Lecture du fichier externe commun V3/V5
        df_raw_v3v5 = pd.read_excel(
            "Data/Matthieu_Soumaya_Dec2025.xlsx",
            sheet_name="UPDRSIII_COMPLET_V3_V5 "
        )
        label_map = {
            3: "Visite Bilan à 3 ans - V3",
            5: "Visite Bilan à 5 ans - V5",
        }
        df_UPDRSIII = df_raw_v3v5[
            df_raw_v3v5["VISIT"] == label_map[v]
        ].reset_index(drop=True)

        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=(".",))
        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=("D",))

        # Conversion numérique des colonnes de score
        cols_num = df_UPDRSIII.iloc[:, 4:].columns
        df_UPDRSIII[cols_num] = df_UPDRSIII[cols_num].apply(pd.to_numeric, errors="coerce")

        # Supprimer les 2 dernières colonnes redondantes
        # df_UPDRSIII.drop(columns=df_UPDRSIII.iloc[:, -2:].columns, inplace=True)

        # Calcul du total (toutes les colonnes de score sauf les sous-totaux)
        cols_score = df_UPDRSIII.iloc[:, 4:-2].columns
        cols_score = cols_score.drop(["SS_TOTAL1","ON_SS_TOT1"], errors="ignore")
        df_UPDRSIII["UPDRSIII_tot"] = df_UPDRSIII[cols_score].sum(axis=1, skipna=True)
        df_UPDRSIII["statut"] = np.where(
            np.isclose(df_UPDRSIII["UPDRSIII_tot"], df_UPDRSIII["ON_TOTAL"], atol=0.01),
            "ok", "différent"
        )
        df_Total_UPDRSIII = df_UPDRSIII[["SUBJID","UPDRSIII_tot"]]
        print(f"\ndf_UPDRSIII (V{v})       : {df_UPDRSIII.shape}")


    cols_v=df_V.iloc[:,-2:].columns
    df_V[cols_v] = df_V[cols_v].apply(pd.to_numeric, errors="coerce")

    # convertir la taille en mètres
    df_V["TAILLE_m"] = np.where(df_V["TAILLE"] > 3,
                          df_V["TAILLE"] / 100,
                          df_V["TAILLE"])


    df_V["IMC"] = df_V["POIDS"] / (df_V["TAILLE_m"] ** 2)
    df_V.drop("TAILLE_m",axis=1,inplace=True)

    # ── UPDRSIV ───────────────────────────────────────────────────────
    df_UPDRSIV = pd.read_excel(chemin, sheet_name="UPDRSIV")
    print(f"\nFeuille UPDRSIV         : {df_UPDRSIV.shape}")
    df_UPDRSIV = nettoyer_codes_manquants(df_UPDRSIV)
    mapping_UPDRSIV = {"Normal": 0, "Minime": 1, "Léger": 2, "Modéré": 3, "Sévère": 4}
    df_UPDRSIV.iloc[:, 3:] = df_UPDRSIV.iloc[:, 3:].replace(mapping_UPDRSIV)
    df_UPDRSIV["UPDRSIV_tot"] = df_UPDRSIV.iloc[:, 2:].sum(axis=1, skipna=True)
    df_Total_UPDRSIV = df_UPDRSIV[["SUBJID", "UPDRSIV_tot"]]
    print(f"df_UPDRSIV              : {df_UPDRSIV.shape}")

    # ── PDQ39 ─────────────────────────────────────────────────────────
    df_PDQ39 = pd.read_excel(chemin, sheet_name="PDQ39")
    print(f"\nFeuille PDQ39           : {df_PDQ39.shape}")
    df_PDQ39 = nettoyer_codes_manquants(df_PDQ39)
    mapping_PDQ39 = {
        "Jamais": 0, "Rarement": 1, "Parfois": 2, "Souvent": 3,
        "Toujours": 4, "Toujours ou ne peut jamais faire": 4,
        "Oui": 1, "Non": 0
    }
    cols_obj = df_PDQ39.select_dtypes(include="object").columns[1:]
    df_PDQ39[cols_obj] = df_PDQ39[cols_obj].replace(mapping_PDQ39)
    cols_pdq = df_PDQ39.columns[df_PDQ39.columns.str.match(r"^PDQ39_\d+$")]
    df_PDQ39["PDQ39_tot"] = df_PDQ39[cols_pdq].sum(axis=1, skipna=True)

    cols = ["PDQ39_tot", "PDQ39_SCORE"]
    df_PDQ39[cols] = df_PDQ39[cols].apply(pd.to_numeric, errors="coerce")

    df_PDQ39["statut"] = np.where(
        np.isclose(df_PDQ39["PDQ39_tot"], df_PDQ39["PDQ39_SCORE"], atol=0.01),
        "ok", "différent"
    )
    df_Total_PDQ39 = df_PDQ39[["SUBJID", "PDQ39_tot"]]
    print(f"df_PDQ39                : {df_PDQ39.shape}")

    # ── QUIP ──────────────────────────────────────────────────────────
    df_QUIP = pd.read_excel(chemin, sheet_name="QUIP")
    print(f"\nFeuille QUIP            : {df_QUIP.shape}")
    df_QUIP = nettoyer_codes_manquants(df_QUIP)

    # ── MOCA ──────────────────────────────────────────────────────────
    df_MOCA = pd.read_excel(chemin, sheet_name="MOCA")
    print(f"\nFeuille MOCA            : {df_MOCA.shape}")
    df_MOCA = nettoyer_codes_manquants(df_MOCA)
    cols_moca = df_MOCA.iloc[:, 2:].columns
    df_MOCA[cols_moca] = df_MOCA[cols_moca].apply(pd.to_numeric, errors="coerce")
    df_MOCA["MOCA_tot"] = df_MOCA.iloc[:, 3:-1].sum(axis=1, skipna=True)
    df_MOCA["statut"] = np.where(
        np.isclose(df_MOCA["MOCA_tot"], df_MOCA["MOCA_SCORE"], atol=0.01),
        "ok", "différent"
    )
    df_Total_MOCA = df_MOCA[["MOCA_tot"]]
    print(f"df_MOCA                 : {df_MOCA.shape}")

    # ── HAMA ──────────────────────────────────────────────────────────
    df_HAMA = pd.read_excel(chemin, sheet_name="HAMA")
    print(f"\nFeuille HAMA            : {df_HAMA.shape}")
    df_HAMA = nettoyer_codes_manquants(df_HAMA)
    mapping_HAMA = {
        "Absent": 0,
        "Léger": 1, "Anxiété légère": 1,
        "Modéré": 2, "Anxiété légère à modérée": 2,
        "Sévère": 3, "Anxiété modérée à grave": 3,
        "Très sévère": 4
    }
    cols_obj = df_HAMA.select_dtypes(include="object").columns[1:]
    df_HAMA[cols_obj] = df_HAMA[cols_obj].replace(mapping_HAMA)


    cols_hama = df_HAMA.iloc[:, 2:-2].columns
    df_HAMA[cols_hama] = df_HAMA[cols_hama].apply(pd.to_numeric, errors="coerce")
    df_HAMA["HAMA_SCORECALC"] = df_HAMA["HAMA_SCORECALC"].apply(pd.to_numeric, errors="coerce")
    df_HAMA["HAMA_tot"] = df_HAMA[cols_hama].sum(axis=1, skipna=True)
    df_HAMA["statut"] = np.where(
        np.isclose(df_HAMA["HAMA_tot"], df_HAMA["HAMA_SCORECALC"], atol=0.01),
        "ok", "différent"
    )
    # Déplacer ANXIETE en dernière colonne
    col = df_HAMA.pop("ANXIETE")
    df_HAMA["ANXIETE"] = col
    df_Total_HAMA = df_HAMA[["SUBJID","HAMA_tot"]]
    print(f"df_HAMA                 : {df_HAMA.shape}")

    # ── HAMD ──────────────────────────────────────────────────────────
    df_HAMD = pd.read_excel(chemin, sheet_name="HAMD")
    print(f"\nFeuille HAMD            : {df_HAMD.shape}")
    df_HAMD = nettoyer_codes_manquants(df_HAMD)
    cols_hamd = df_HAMD.iloc[:, 2:-2].columns
    df_HAMD["HAMD_tot"] = df_HAMD[cols_hamd].sum(axis=1, skipna=True)
    df_HAMD["statut"] = np.where(
        np.isclose(df_HAMD["HAMD_tot"], df_HAMD["HAMD_SCORECALC"], atol=0.01),
        "ok", "différent"
    )
    col = df_HAMD.pop("DEPRESSION")
    df_HAMD["DEPRESSION"] = col
    mapping_HAMD = {
        "Symptomes dépressifs légers": 1,
        "Symptômes dépressifs légers à modérés": 2,
        "Symptômes dépressifs modérés à sévères": 3,
    }
    cols_obj = df_HAMD.select_dtypes(include="object").columns[-1:]
    df_HAMD[cols_obj] = df_HAMD[cols_obj].replace(mapping_HAMD)
    df_Total_HAMD = df_HAMD[["SUBJID", "HAMD_tot"]]
    print(f"df_HAMD                 : {df_HAMD.shape}")

    # ── LARS ──────────────────────────────────────────────────────────
    df_LARS = pd.read_excel(chemin, sheet_name="LARS")
    print(f"\nFeuille LARS            : {df_LARS.shape}")
    df_LARS = nettoyer_codes_manquants(df_LARS)
    cols_lars = df_LARS.iloc[:, 2:-2].columns
    df_LARS["LARS_tot"] = df_LARS[cols_lars].sum(axis=1, skipna=True)
    df_LARS["statut"] = np.where(
        np.isclose(df_LARS["LARS_tot"], df_LARS["LARS_SCORE"], atol=0.01),
        "ok", "différent"
    )
    col = df_LARS.pop("LARS_RESULTAT")
    df_LARS["LARS_RESULTAT"] = col
    mapping_LARS = {
        "Non apathique": 1,
        "Tendance à l'apathie": 2,
        "Apathie modérée": 3,
        "Apathie sévère": 4,
    }
    cols_obj = df_LARS.select_dtypes(include="object").columns[-1:]
    df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)
    df_Total_LARS = df_LARS[["SUBJID", "LARS_tot"]]
    print(f"df_LARS                 : {df_LARS.shape}")

    # ── ECMP ──────────────────────────────────────────────────────────
    df_ECMP = pd.read_excel(chemin, sheet_name="ECMP")
    print(f"\nFeuille ECMP            : {df_ECMP.shape}")
    df_ECMP = nettoyer_codes_manquants(df_ECMP)

    # ── DIGITSMT_TRAILMT_DKEFS ────────────────────────────────────────
    df_DIGITSMT = pd.read_excel(chemin, sheet_name="DIGITSMT_TRAILMT_DKEFS")
    print(f"\nFeuille DIGITSMT        : {df_DIGITSMT.shape}")
    df_DIGITSMT = nettoyer_codes_manquants(df_DIGITSMT)

    # ── FREQUENCE  (V1, V3, V5) ───────────────────────────────────────
    if v in (1, 2, 3):
        df_FREQUENCE = pd.read_excel(chemin, sheet_name="FREQUENCE")
        df_FREQUENCE = nettoyer_codes_manquants(df_FREQUENCE)
       

    # ==================================================================
    # TABLE DES TOTAUX  (merge de tous les scores résumés)
    # ==================================================================
    lis_totaux = [
        df_Total_LEDD,
        df_Total_UPDRSIII,
        df_Total_UPDRSIV,
        df_Total_PDQ39,
        df_Total_HAMA,
        df_Total_HAMD,
        df_Total_LARS,
    ]
    # Seuls les DataFrames qui ont une colonne SUBJID participent au merge
    lis_avec_id = [d for d in lis_totaux if "SUBJID" in d.columns]
    if lis_avec_id:
        df_Totaux = reduce(
            lambda left, right: pd.merge(left, right, on="SUBJID", how="outer"),
            lis_avec_id
        )
        # Remonter la ligne de description (dernière ligne) en tête
        df_Totaux = pd.concat(
            [df_Totaux.iloc[[-1]], df_Totaux.iloc[:-1]], ignore_index=True
        )
    else:
        df_Totaux = pd.DataFrame()

    # df_Totaux["IMC"]=df_V["IMC"]
    # ==================================================================
    # CONSTRUCTION DU DICTIONNAIRE DE RETOUR
    # ==================================================================
    result = {
        f"V{v}"                   : df_V,
        "LEDD"                    : df_LEDD,
        "CONSO_SPECIFIQUE"        : df_CONSO_SPECIFIQUE,
        "PSYCHOTROPES"            : df_PSYCHOTROPES,
        "AUTRE_PARKINSON"         : df_AUTRE_PARKINSON,
        "UPDRSIII"                : df_UPDRSIII,
        "UPDRSIV"                 : df_UPDRSIV,
        "PDQ39"                   : df_PDQ39,
        "QUIP"                    : df_QUIP,
        "MOCA"                    : df_MOCA,
        "HAMA"                    : df_HAMA,
        "HAMD"                    : df_HAMD,
        "LARS"                    : df_LARS,
        "ECMP"                    : df_ECMP,
        "DIGITSMT_TRAILMT_DKEFS"  : df_DIGITSMT,
        "Totaux"                  : df_Totaux,
    }

    # Bloc FREQUENCE (V1, V2, V3) —
    if v in (1, 2, 3):
        result["FREQUENCE"] = df_FREQUENCE

    return result

---
## Utilitaire d'écriture Excel

In [ ]:
def ecrire_excel(sheets_dict, version, output_dir="Output/version_3"):
    filepath = f"{output_dir}/V{version}.xlsx"
    with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
        for sheet_name, df in sheets_dict.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"[OK] {filepath}  →  {len(sheets_dict)} feuilles")

---
## Traitement + écriture de chaque visite

### Vc

In [ ]:
sheets_Vc = traiter_visite("c", vc=True)
ecrire_excel(sheets_Vc, "c")

### V0

In [ ]:
sheets_V0 = traiter_visite(0)
ecrire_excel(sheets_V0, 0)

### V1

In [ ]:
sheets_V1 = traiter_visite(1)
ecrire_excel(sheets_V1, 1)

### V3

In [ ]:
sheets_V3 = traiter_visite(3)
ecrire_excel(sheets_V3, 3)

### V5

In [ ]:
sheets_V5 = traiter_visite(5)
ecrire_excel(sheets_V5, 5)

# Prétraitement info statiques 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
info =  pd.read_excel("Output/version_2/info.xlsx")
info.head()


In [ ]:
info.drop("D_SCREEN",axis=1,inplace=True)
info = nettoyer_codes_manquants(info)
info.to_excel("Output/version_3/info.xlsx",index=False)
